# Demo 2: SCD2 + Fact tábla típusok
## Slowly Changing Dimensions és dimenziós modellezés DuckDB-ben

**Cél:** Megértjük, hogyan kezeljük az adatok időbeli változásait a DWH-ban (SCD), és megismerjük a három különböző fact tábla típust.

**Kapcsolódó diák:** 13–22 (Dimenzió típusok, SCD, Grain declaration, Fact típusok)

## 0. Előkészítés

Ebben a demóban csak **DuckDB**-t és **pandas**-t használunk – nincs szükség PostgreSQL-re.
DuckDB helyi, in-memory adatbázisként működik, ami kifejezetten gyors prototipizálásra.

### Technológia: DuckDB in-memory kapcsolat

A `duckdb.connect()` egy **session-scoped** memória adatbázist hoz létre – nincs fájl, nincs szerver.
Az adatok a kernel futása alatt megmaradnak. Ez ideális prototipizáláshoz és oktatáshoz.


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "duckdb", "pandas"], check=True)

import duckdb
import pandas as pd
from datetime import date, timedelta

con = duckdb.connect()  # helyi, memóriában tárolt adatbázis
print(f"DuckDB {duckdb.__version__} kész.")

## 1. rész: SCD Type 1 vs Type 2 – az alapkérdés

**Mi az SCD?**

A dimenziókban lévő adatok az idő múlásával változnak:
- Az ügyfél elköltözik → új városba kerül
- Az értékesítő más régiót kap → más szegmensbe kerül
- A termék más kategóriába sorolódik

**SCD Type 1** – *felülírás*: az új érték felülírja a régit. A múlt **elveszik**.

**SCD Type 2** – *új sor*: a régi sor megmarad, az új sor keletkezik `valid_from`/`valid_to` dátumokkal. A múlt **megőrződik**.

A kérdés: *Meg kell-e tudnunk kérdezni, hogy az adott ügyfél hol lakott 2022-ben?*
Ha igen → SCD2. Ha nem → SCD1 elég.

### Adatmérnöki döntés: SCD1 vs SCD2 táblaszerkesztés

Az SCD1 tábla **egyszerűbb**: `customer_id` az egyetlen kulcs, egy sor per ügyfél.
Az SCD2 tábla **bonyolultabb** három kötelező oszloppal:
- `customer_sk` – surrogate kulcs (minden verzió külön SK-t kap)
- `valid_from / valid_to` – érvényességi időszak
- `is_current` – gyors szűrő az aktuális rekordhoz

**Miért `9999-12-31` a nyitott `valid_to` értéke?**
Ez az iparágban elterjedt konvenció a "végtelenül érvényes" rekord jelzésére.
Alternatíva a `NULL`, de az nehezebben indexelhető és BETWEEN-ben awkward.


In [ ]:
# SCD1 tábla: egyszerű – egy sor per ügyfél, PK = customer_id (természetes kulcs)
con.execute("""
CREATE OR REPLACE TABLE dim_customer_scd1 (
    customer_id INTEGER PRIMARY KEY,  -- természetes kulcs
    name        VARCHAR,
    city        VARCHAR,
    segment     VARCHAR
)
""")

# SCD2 tábla: összetettebb – TÖBB sor is lehet per ügyfél (verziók!)
# customer_sk = surrogate kulcs (egyedi minden verzióhoz)
# customer_id = természetes kulcs (ismétlődhet, ha van több verzió)
# valid_from/valid_to: a sor érvényességi ablaka
# is_current: gyors szűrés az aktuális verzióhoz
con.execute("""
CREATE OR REPLACE TABLE dim_customer_scd2 (
    customer_sk  INTEGER,  -- surrogate kulcs (egyedi minden sorhoz)
    customer_id  INTEGER,  -- természetes kulcs (ismétlődhet!)
    name         VARCHAR,
    city         VARCHAR,
    segment      VARCHAR,
    valid_from   DATE,     -- mikortól érvényes ez a verzió
    valid_to     DATE,     -- meddig érvényes (9999-12-31 = aktuális)
    is_current   BOOLEAN   -- gyors szűrő flag
)
""")
print("Táblák létrehozva: dim_customer_scd1, dim_customer_scd2")

### Kezdeti betöltés (Initial Load)

A kezdeti betöltésnél minden ügyfél egy "első verziót" kap:
`valid_from = 2020-01-01`, `valid_to = 9999-12-31`, `is_current = TRUE`.

**Adatmérnöki döntés:** a surrogate key (`customer_sk`) értékei az SCD1 és SCD2 táblákban
szándékosan különbözők lehetnek – az SCD2-t soha nem szabad az SCD1 SK-jával összekeverni.


In [ ]:
# Kezdeti adatok betöltése – 3 ügyfél
# SCD2-ben kezdetben is: valid_from = 2020-01-01, valid_to = 9999-12-31, is_current = TRUE
customers = [
    (1, "Kiss József",  "Pécs",    "Consumer"),
    (2, "Nagy Mária",   "Budapest","Corporate"),
    (3, "Kovács Péter", "Győr",    "Consumer"),
]

for cid, name, city, seg in customers:
    # SCD1: egyszerű INSERT
    con.execute("INSERT INTO dim_customer_scd1 VALUES (?,?,?,?)", [cid, name, city, seg])
    # SCD2: az sk = cid-del indul (de változásnál SK különböző lesz)
    con.execute("INSERT INTO dim_customer_scd2 VALUES (?,?,?,?,?,?,?,?)",
                [cid, cid, name, city, seg, date(2020,1,1), date(9999,12,31), True])

print("Kezdeti állapot – SCD1 (egy sor per ügyfél):")
print(con.execute("SELECT * FROM dim_customer_scd1").df().to_string(index=False))
print("\nKezdeti állapot – SCD2 (szintén egy sor, is_current=True):")
print(con.execute("SELECT * FROM dim_customer_scd2").df().to_string(index=False))

## 2. rész: Változás alkalmazása

Kiss József 2023. július 1-jén Pécsről Debrecenbe költözik.

**SCD1 hatás:** az egyetlen sor felülíródik. *Pécs eltűnik.*  
**SCD2 hatás:**
1. A régi sor `valid_to` értékét beállítjuk `2023-06-30`-ra (`is_current = FALSE`)
2. Egy új sort szúrunk be `valid_from = 2023-07-01`, `is_current = TRUE`

### A két SCD függvény – az SCD2 kétlépéses mintája

`apply_scd1_change`: egyszerű `UPDATE` – a régi érték felülíródik és **elvész**.
Ezt csak adathibára vagy üzletileg irreleváns változásra alkalmazzuk.

`apply_scd2_change`: a **két kötelező lépés**:
1. Régi sor lezárása: `valid_to = change_date`, `is_current = FALSE`
2. Új sor megnyitása: `valid_from = change_date`, `valid_to = 9999-12-31`, `is_current = TRUE`

**Technológia:** A sorrend fontos! Ha előbb INSERT, aztán UPDATE, az egyszerre két
`is_current = TRUE` sort eredményez átmenetileg – tranzakcióban kell futtatni prodban.


In [ ]:
def apply_scd1_change(con, customer_id: int, new_city: str):
    """SCD Type 1: egyszerű UPDATE – a régi érték felülíródik és elvész."""
    con.execute(
        "UPDATE dim_customer_scd1 SET city=? WHERE customer_id=?",
        [new_city, customer_id]
    )


def apply_scd2_change(con, customer_id: int, new_city: str, change_date: date, next_sk: int):
    """SCD Type 2: régi sor lezárása + új sor létrehozása."""
    # 1. lépés: a jelenlegi 'current' sort lezárjuk
    #    valid_to = change_date - 1 nap (az előző nap volt az utolsó érvényes)
    con.execute("""
        UPDATE dim_customer_scd2
        SET valid_to   = ?,
            is_current = FALSE
        WHERE customer_id = ? AND is_current = TRUE
    """, [change_date - timedelta(days=1), customer_id])

    # 2. lépés: a régi sorból kiolvassuk, amit nem változtattunk (name, segment)
    prev = con.execute(
        "SELECT name, segment FROM dim_customer_scd2 WHERE customer_id=? ORDER BY valid_from DESC LIMIT 1",
        [customer_id]
    ).fetchone()

    # 3. lépés: új sort szúrunk be az új SK-val, új várossal, today-től 9999-ig
    con.execute("INSERT INTO dim_customer_scd2 VALUES (?,?,?,?,?,?,?,?)",
                [next_sk, customer_id, prev[0], new_city, prev[1],
                 change_date, date(9999,12,31), True])


print("SCD1 és SCD2 függvények definiálva.")

### SCD1 vs SCD2 – a különbség vizuálisan

Az alábbi kimenetből látható:
- **SCD1**: Kiss József egyetlen sorában `city = 'Debrecen'` – Pécs eltűnt
- **SCD2**: Kiss Józsefnek **két sora** van – az első lezárt (Pécs), a második aktív (Debrecen)

Ez az alapvető különbség: SCD1-gyel elveszítjük a múltat, SCD2-vel megőrizzük.


In [ ]:
# Kiss József elköltözik: Pécs → Debrecen (2023-07-01)
apply_scd1_change(con, customer_id=1, new_city="Debrecen")
apply_scd2_change(con, customer_id=1, new_city="Debrecen", change_date=date(2023,7,1), next_sk=101)

print("SCD1 – Kiss József most (Pécs eltűnt!):")
print(con.execute("SELECT * FROM dim_customer_scd1 WHERE customer_id=1").df().to_string(index=False))

print("\nSCD2 – Kiss József verziói (mindkét állapot megőrizve):")
print(con.execute(
    "SELECT customer_sk, customer_id, name, city, valid_from, valid_to, is_current FROM dim_customer_scd2 WHERE customer_id=1 ORDER BY valid_from"
).df().to_string(index=False))
print("\n→ customer_sk=1 (Pécs), customer_sk=101 (Debrecen) – két különböző SK, egy természetes kulcs!")

## 3. rész: Kettős változás – 3 verzió

Nagy Mária kétszer költözik:
1. 2022-03-15: Budapest → Miskolc
2. 2024-01-01: Miskolc → Debrecen

Eredmény: **3 sor** a dim_customer_scd2 táblában, 3 különböző SK-val.

### Kettős változás – 3 sor az SCD2-ben

Minden `apply_scd2_change()` hívás egy újabb sort szúr be.
Nagy Mária esetén 3 verzió keletkezik, mindegyik **más surrogate key-jel** (`customer_sk`).

**Miért fontos ez a fact táblánál?**
Ha Nagy Mária 2022 szeptemberében (Miskolcon) vásárolt, a fact táblában
a Miskolc-os `customer_sk` szerepel. A JOIN visszahozza a Miskolc-os sort.
Az analitika helyesen mutatja: ez a vásárlás "Miskolc" régióhoz tartozik.


In [ ]:
# Kettős változás Nagy Máriánál
apply_scd2_change(con, customer_id=2, new_city="Miskolc",  change_date=date(2022,3,15), next_sk=102)
apply_scd2_change(con, customer_id=2, new_city="Debrecen", change_date=date(2024,1,1),  next_sk=103)

print("Nagy Mária – 3 verzió:")
print(con.execute("""
    SELECT customer_sk, name, city, valid_from, valid_to, is_current
    FROM dim_customer_scd2
    WHERE customer_id = 2
    ORDER BY valid_from
""").df().to_string(index=False))

### SCD2 visszakérdezés – a dátumtartomány-szűrő

Ez a lekérdezési minta **kötelező** SCD2 esetén. A feltétel:
```sql
WHERE customer_id = 2
  AND valid_from <= '2023-06-01'
  AND valid_to   >  '2023-06-01'
```
**Miért `valid_to > dátum` (nem `>=`)?**
Ha a változás dátuma 2022-03-15, akkor:
- Régi sor: `valid_to = 2022-03-15` (az utolsó érvényes nap = a változás napja)
- Új sor:   `valid_from = 2022-03-15`

A `>` operátorral biztosítjuk, hogy pontosan egy sor adja vissza az eredményt: nincs átfedés, nincs rés.


In [ ]:
# Visszakérdezés: 2023-06-01-én hol lakott Nagy Mária?
# A WHERE feltétel: valid_from <= kérdés dátuma < valid_to
# Ez pontosan megadja, melyik sor volt érvényes az adott napon
check_date = "2023-06-01"

result = con.execute(f"""
    SELECT customer_sk, name, city, segment, valid_from, valid_to
    FROM dim_customer_scd2
    WHERE customer_id = 2
      AND valid_from <= DATE '{check_date}'   -- a verzió már elindult
      AND valid_to   >  DATE '{check_date}'   -- a verzió még nem zárult le
""").df()

print(f"Nagy Mária aktuális állapota {check_date}-én:")
print(result.to_string(index=False))
print("\n→ Miskolcon lakott, mert 2022-03-15 – 2023-12-31 között volt a miskolci verzió érvényes.")

## 4. rész: Fact tábla típusok

Háromféle fact tábla létezik, amelyek különböző üzleti kérdésekre adnak választ:

| Típus | Mit tárol? | Grain | Módosítás |
|-------|-----------|-------|----------|
| **Transaction** | Minden eseményt | 1 esemény = 1 sor | INSERT only |
| **Periodic Snapshot** | Időszaki állapotot | 1 időszak × 1 entitás = 1 sor | INSERT havonta |
| **Accumulating Snapshot** | Folyamat életciklusát | 1 folyamat = 1 sor | INSERT + UPDATE |

### Transaction Fact tábla – a legelterjedtebb típus

A tranzakciós ténytábla az SCD2-vel kombinálva megőrzi a historikus kontextust:
a `customer_sk` az **akkori** ügyfél-verzióra mutat, nem a mostanira.

**Technológia:** a `date_sk` integer (`YYYYMMDD`) formátum azért előnyös, mert:
- BETWEEN szűrés integereken gyorsabb, mint DATE típuson
- Vizuálisan olvasható: `20240115` = 2024. január 15.
- Particionáláshoz is jól használható


In [ ]:
# ===== TRANSACTION FACT =====
# Minden egyes vásárlási esemény egy sor
# Grain: 1 rendelési tétel (order × termék)
# Kérdés: 'Mennyi volt a bevétel 2024. január 10-én?'
con.execute("""
CREATE OR REPLACE TABLE fact_transactions AS
SELECT * FROM (VALUES
    (1, 20240101, 101, 201, 3, 49.99,  149.97),  -- Kiss József 3 db terméket vett
    (2, 20240103, 102, 205, 1, 299.0,  299.0),
    (3, 20240110, 101, 201, 2, 49.99,   99.98),
    (4, 20240115, 103, 210, 5, 15.0,    75.0),
    (5, 20240201, 101, 201, 1, 49.99,   49.99),
    (6, 20240205, 102, 207, 2, 89.9,   179.8)
) t(order_line_sk, date_sk, customer_sk, product_sk, quantity, unit_price, revenue)
""")

print("Transaction Fact – minden esemény egy sor:")
print(con.execute("SELECT * FROM fact_transactions").df().to_string(index=False))
print(f"\nGrain: 1 sor = 1 rendelési tétel | {con.execute('SELECT COUNT(*) FROM fact_transactions').fetchone()[0]} sor összesen")

### Transaction Fact analitika

A tranzakciós ténytáblából bármilyen időszak összesíthető: nap, hét, hónap, negyedév.
Ez a rugalmasság az elsődleges előnye a Periodic Snapshot-hoz képest.


In [ ]:
# Analitikai lekérdezés a Transaction Fact táblán
# Kérdés: melyik ügyfél mennyi bevételt hozott januárban?
result = con.execute("""
    SELECT customer_sk,
           SUM(revenue)   AS total_revenue,
           SUM(quantity)  AS total_qty,
           COUNT(*)       AS num_transactions
    FROM fact_transactions
    WHERE date_sk BETWEEN 20240101 AND 20240131
    GROUP BY customer_sk
    ORDER BY total_revenue DESC
""").df()
print("Januári bevétel ügyfelenként:")
print(result.to_string(index=False))

### Periodic Snapshot Fact – az egyenleg-minta

A periodikus snapshot **időszak végén fennálló állapotot** rögzít.

**Fontos különbség a Transaction Fact-tól:**
Ha egy ügyfélnek januárban 0 rendelése volt, a Periodic Snapshot-ban **mégis szerepel**
egy sor (0 értékekkel). A Transaction Fact-ból ez nem lenne rekonstruálható.

**Tipikus felhasználás:** készletszint hónap végén, bankszámlaegyenleg naponta,
előfizetők száma hetente.


In [ ]:
# ===== PERIODIC SNAPSHOT FACT =====
# Havi pillanatkép: mit mutat az egyenleg az időszak végén?
# Grain: 1 sor = 1 ügyfél × 1 hónap
# Kérdés: 'Mekkora volt az ügyfél összes rendelése január végén?'
# Ez NEM esemény-szintű – ez állapot a hónap végén!
con.execute("""
CREATE OR REPLACE TABLE fact_monthly_snapshot AS
SELECT * FROM (VALUES
    (202401, 101, 3, 299.95, 2),   -- Kiss József: 3 rendelés, 299.95 bevétel januárban
    (202401, 102, 1, 299.0,  1),
    (202401, 103, 5, 75.0,   1),
    (202402, 101, 4, 349.94, 3),   -- Február: +1 rendelés, kumulatív szinten mutatja
    (202402, 102, 3, 479.7,  2)
) t(month_sk, customer_sk, total_orders, total_revenue, active_products)
""")

print("Periodic Snapshot – havi egyenleg (nem egyes események!):")
print(con.execute("SELECT * FROM fact_monthly_snapshot").df().to_string(index=False))
print("\nGrain: 1 sor = 1 ügyfél × 1 hónap | Frissítés: havonta új sorok kerülnek be")

### Accumulating Snapshot – az egyetlen UPDATE-elhető fact tábla

Ez a **kivétel** a DWH immutabilitási szabálya alól. Az akkumuláló snapshot
egyetlen sorban tárolja egy folyamat összes mérföldkövét.

**Adatmérnöki döntés:** mikor válasszuk?
- Rendelés-feldolgozás: feladás → csomagolás → szállítás → kézbesítés
- Pályázati folyamat: benyújtás → bírálat → döntés → értesítés
- Hiteligénylés: kérelem → ellenőrzés → jóváhagyás → folyósítás

**Technológia:** az `order_id` természetes kulcsként szolgál – ez az egyetlen jól ismert
DWH eset, ahol nem surrogate kulcsra joinolunk a fact táblán belül.


In [ ]:
# ===== ACCUMULATING SNAPSHOT FACT =====
# Egy folyamat (pl. rendelés) életciklusa egyetlen sorban
# Grain: 1 sor = 1 rendelés teljes életciklusa
# Milestone dátumok: mikor adták le, mikor csomagolták, mikor szállították, mikor érkezett
# Ha egy milestone még nem történt meg: NULL
con.execute("""
CREATE OR REPLACE TABLE fact_order_lifecycle AS
SELECT * FROM (VALUES
    (1001, 20240101, 20240102, 20240105, NULL,      'shipped'),    -- vár szállításra
    (1002, 20240103, 20240104, NULL,     NULL,      'processing'), -- még csomagolás alatt
    (1003, 20240110, 20240110, 20240112, 20240115,  'delivered')   -- teljes életciklus
) t(order_id, placed_date_sk, packed_date_sk, shipped_date_sk, delivered_date_sk, status)
""")

print("Accumulating Snapshot – rendelés életciklus:")
print(con.execute("SELECT * FROM fact_order_lifecycle").df().to_string(index=False))
print("\n→ A NULL értékek azt jelzik, hogy az adott milestone még nem történt meg.")

### UPDATE az Accumulating Snapshot-ban

A `shipped_date_sk` mező frissül, amikor az esemény bekövetkezik.
Addig `NULL` – a `NULL` az "még nem történt meg" szemantikát hordozza.

**Technológia:** CASE WHEN az átfutási idő számításánál:
```sql
CASE WHEN shipped_date_sk IS NOT NULL
     THEN shipped_date_sk - placed_date_sk
     ELSE NULL
END AS days_to_ship
```
Ez különbözteti meg a "0 napos átfutást" a "még nem szállítottól".


In [ ]:
# Az Accumulating Snapshot egyedülálló jellemzője: UPDATE is megengedett!
# Ha a 1002-es rendelés kiszállítva, frissítjük a sort
# (DWH-ban általában tilos az UPDATE, de itt ez a működési modell)
con.execute("""
    UPDATE fact_order_lifecycle
    SET shipped_date_sk = 20240106,
        status          = 'shipped'
    WHERE order_id = 1002
""")

print("1002-es rendelés frissítve (kiszállítva!):")
print(con.execute("SELECT * FROM fact_order_lifecycle").df().to_string(index=False))
print("\n→ Az UPDATE a milestone dátumon kívül mindent érintetlenül hagy.")

### Átfutási idő elemzés – az Accumulating Snapshot értéke

Ez az a kérdés, amelyre **csak** az Accumulating Snapshot tud választ adni:
*"Átlagosan hány nap telik el a rendelés feladásától a kézbesítésig?"*

A Transaction Fact-ból ez nem számítható: az egyes mérföldkövek külön sorok lennének,
és az összekapcsolásuk bonyolult pivot lekérdezést igényelne.


In [ ]:
# Átfutási idő számítása az Accumulating Snapshot-ból
# Kérdés: átlagosan hány nap telik el a leadástól a szállításig?
result = con.execute("""
    SELECT
        order_id,
        placed_date_sk,
        shipped_date_sk,
        delivered_date_sk,
        -- Szállítási átfutás: placed → shipped (nap)
        CASE WHEN shipped_date_sk IS NOT NULL
             THEN shipped_date_sk - placed_date_sk
             ELSE NULL
        END AS days_to_ship,
        status
    FROM fact_order_lifecycle
    ORDER BY order_id
""").df()

print("Átfutási idő elemzés:")
print(result.to_string(index=False))
print(f"\nÁtlagos szállítási idő: {result['days_to_ship'].mean():.1f} nap")

### Összesítő – Fact tábla döntési mátrix

Az alábbi táblázat segít eldönteni, melyik típust érdemes alkalmazni egy konkrét üzleti kérdésnél.


In [ ]:
# Összesítő – Fact tábla típusok és SCD összehasonlítása
print("=" * 70)
print("ÖSSZESÍTŐ: Fact tábla típusok")
print("=" * 70)

summary = pd.DataFrame({
    "Típus":       ["Transaction Fact", "Periodic Snapshot",   "Accumulating Snapshot"],
    "Mit tárol?":  ["Minden eseményt",   "Időszak végi állapot","Folyamat mérföldköveit"],
    "Grain":       ["1 esemény = 1 sor", "1 id. × 1 entitás",  "1 folyamat = 1 sor"],
    "Módosítás":   ["INSERT only",       "INSERT/időszak",      "INSERT + UPDATE"],
    "Példa":       ["Rendelési tétel",   "Havi forgalom",       "Rendelés életciklus"],
})
print(summary.to_string(index=False))

print("\n" + "=" * 70)
print("ÖSSZESÍTŐ: SCD típusok")
print("=" * 70)
scd_summary = pd.DataFrame({
    "Típus": ["SCD1", "SCD2", "SCD3", "SCD6"],
    "Stratégia": ["Felülírás", "Új sor + valid_from/to", "prev_value oszlop", "SCD1+2+3 kombinált"],
    "Előzmény megőrzés": ["Nincs", "Teljes", "1 előző", "Teljes"],
    "Komplexitás": ["Alacsony", "Közepes", "Alacsony", "Magas"],
})
print(scd_summary.to_string(index=False))